In [0]:
%sql

create or replace table workspace.silver.meter_readings
comment '''
    one row per home and local hour, latest version only. notes:
    (1) local time has no offset: converted to utc. on the dst fall-back the repeated 02:00 is treated as one hour. registered and left alone. 
    (2) hanport_v2 reports wh: divided by 1000. 
    (3) delayed readings - up to 20+ days late: latest ingested_at_utc wins, is_corrected = 1, all versions stay in bronze. 
    (4) missing hours exist: not imputed, completeness could be flagged in the monthly summary.
'''
as

select
    m.home_id,
    cast(m.measured_at_local as timestamp) as measured_at_local,
    to_utc_timestamp(m.measured_at_local, 'Europe/Stockholm') as measured_at_utc,
    cast(cast(m.kwh as double) / if(m.source = 'hanport_v2', 1000, 1) as decimal(12, 3)) as kwh,
    m.source,
    cast(m.ingested_at_utc as timestamp) as ingested_at_utc,
    if(count(*) over w > 1, 1, 0) as is_corrected,
    current_timestamp() as _processed_at
from 
    workspace.bronze.meter_readings as m
window 
    w as (partition by m.home_id, m.measured_at_local)
qualify 
    row_number() over (partition by m.home_id, m.measured_at_local order by m.ingested_at_utc desc) = 1